# MODEL: Evaluation and Baseline Performance

In [1]:
import subprocess
import sys
import os

def get_repo_root():
    return subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).decode('utf-8').strip()
repo_root = get_repo_root()
print(repo_root)

src_path = os.path.join(repo_root, 'src')
# Add src_path to sys.path if not already present
if src_path not in sys.path:
    sys.path.insert(0, src_path)

/home/sagemaker-user/aai-540-su25-group4


In [2]:
# try importing src/utils
from utils.utils import parse_s3_uri
from utils.utils import generate_manifest_file


In [3]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import pandas as pd
import json
from sagemaker.transformer import Transformer


role = get_execution_role()
region = boto3.Session().region_name
s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker")
sess = sagemaker.Session()

# project bucket
bucket_name = "aai-540-data"

# provide validation and test s3 folders,
s3_validation_path = f"s3://{bucket_name}/dev_split/validation"
val_key = "val-meta.csv"

s3_test_path = f"s3://{bucket_name}/dev_split/test"
test_key = "test-meta.csv"

# provide s3 full path to label_mapping.json use during training
s3_label_map_uri = f"s3://{bucket_name}/dev_split/label_mapping.json"


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [4]:
# Load Model Package arn from notebook 03.
from sagemaker import ModelPackage
# 1. Create Model resource from package ARN
model = ModelPackage(
    model_package_arn='arn:aws:sagemaker:us-east-1:324183265896:model-package/wildscan-image-classifiers/1',
    role=role,
    sagemaker_session=sess
)

In [5]:
model.create()

In [ ]:
# Generate Manifest File for Validation and Test Meta CSVs for Evaluation Via Batch Transform
generate_manifest_file(s3_input_csv = f"{s3_validation_path}/{val_key}", s3_images_loc = f"s3://{bucket_name}/cct_resized/")
generate_manifest_file(s3_input_csv = f"{s3_test_path}/{test_key}", s3_images_loc = f"s3://{bucket_name}/cct_resized/")


-----
### Batch Transform Validation Set, then Evaluate Performance

In [6]:
# Transform the Validation Set First

s3_transform_manifest = f"{s3_validation_path}/val-meta.manifest"
s3_transform_out = f"{s3_validation_path}/batch_transform_out"

# initialize Tranformer
transformer = Transformer(
    model_name = model.name,
    instance_count=1,  # Number of instances
    instance_type="ml.g4dn.xlarge",  # Instance type
    output_path= s3_transform_out,  # Predictions output
    max_payload=10,  # Max payload size (MB)
    strategy="MultiRecord" , # for faster processing, but in real world, instance type can be ml.m5.xlarge and single record strategy is ok
    max_concurrent_transforms=10,
    sagemaker_session=sess,

    accept = 'txt/csv', # so output is generated in single file
    assemble_with='Line', # new line is generated for each prediction

)

In [7]:
# Transform the Validation Set

# batch transform images in manifest file
transformer.transform(
    data=s3_transform_manifest,
    data_type='ManifestFile', # provide list of s3uris of objects to be batch transformed
    content_type='application/x-image', 
    split_type='None', # because each object is an image file to be processed, no splitting needed
    logs=True,
    wait=True
)


INFO:sagemaker:Creating transform job with name: wildscan-image-classifiers-2025-06-23-1-2025-06-23-19-52-35-770


......................................Docker entrypoint called with argument(s): serve
Running default environment configuration script
Nvidia gpu devices, drivers and cuda toolkit versions (only available on hosts with GPU):
Mon Jun 23 19:58:58 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.256.02   Driver Version: 470.256.02   CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            On   | 00000000:00:1E.0 Off |                    0 |
| N/A   34C    P8    11W /  70W |      0MiB / 15109MiB |      0%      Default |
|                     

In [8]:
# Evaluate the Prediction Results via ScriptProcessing
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

image_uri = sagemaker.image_uris.retrieve(
    framework='sklearn',        # or 'xgboost', 'pytorch', etc.
    region=region,
    version='1.2-1',            # Specify the version you need
    py_version='py3',           # Specify Python version if required
       # Use 'processing' for processing jobs
)

print(image_uri)

# Define your processing container (can use a built-in or custom image)
script_processor = ScriptProcessor(
    command=['python3'],
    image_uri=image_uri,  # e.g., a scikit-learn or custom image
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    
)

INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3


In [9]:

# Run evaluation script
s3_evaluation_out = f"{s3_validation_path}/evaluation"
s3_true_meta_uri = f"{s3_validation_path}/{val_key}"

script_processor.run(
    code='../src/evaluation/evaluate.py',  # Your processing script
    inputs=[
        # S3 location of batch transform predictions files
        ProcessingInput(
            source=s3_transform_out,       # S3 bucket with predictions
            destination='/opt/ml/processing/input_predictions'        # Where the script will read input
        ),
        
        # S3 location of the ground truth labels for the images in this set
        ProcessingInput(
            source=s3_true_meta_uri,
            destination='/opt/ml/processing/true_labels'
        ),

        # Label Mapping
        ProcessingInput(
            source=s3_label_map_uri,
            destination='/opt/ml/processing/label_mapping'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',           # Where the script will write output
            destination=s3_evaluation_out    # S3 bucket to store results
        )
    ]
)

INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2025-06-23-20-03-24-272


..............[2025-06-23 20:05:38.949123] Script has started.
INFO:root:Script has started at 2025-06-23 20:05:38.949164.
Number of files in /opt/ml/processing/input_predictions: 6048
pred probs df shape:(6048, 2)
 (step 1) pred probs df shape: (6048, 2)
 (step 2) True Labels data shape: (6048, 8)
 (step3) Merged data shape : (6048, 9)
 (step4) Merged data shape with preds: (6048, 11)
 (step5) METRICS CALCULATION
 (step5.1) Class-Restricted metrics - Evaluate only on labels available during training
 (step5.1) class_restriced df shape : (6048, 11)
              precision    recall  f1-score   support
        bird       0.62      0.81      0.70        26
      bobcat       0.76      0.86      0.81       454
         car       1.00      1.00      1.00       424
         cat       0.69      0.81      0.75       511
      coyote       0.79      0.90      0.84       668
        deer       1.00      0.80      0.89         5
         dog       0.90      0.75      0.82       380
       empty 

-----
### Batch Transform Test Set, then Evaluate Performance

In [11]:
# Transform the Test Set
s3_transform_manifest = f"{s3_test_path}/test-meta.manifest"
s3_transform_out = f"{s3_test_path}/batch_transform_out"

# initialize Tranformer
transformer1 = Transformer(
    model_name = model.name,
    instance_count=1,  # Number of instances
    instance_type="ml.g4dn.xlarge",  # Instance type
    output_path= s3_transform_out,  # Predictions output
    max_payload=10,  # Max payload size (MB)
    strategy="MultiRecord" , # for faster processing, but in real world, instance type can be ml.m5.xlarge and single record strategy is ok
    max_concurrent_transforms=10,
    sagemaker_session=sess,

    accept = 'txt/csv', # so output is generated in single file
    assemble_with='Line', # new line is generated for each prediction

)

# batch transform images in manifest file
transformer1.transform(
    data=s3_transform_manifest,
    data_type='ManifestFile', # provide list of s3uris of objects to be batch transformed
    content_type='application/x-image', 
    split_type='None', # because each object is an image file to be processed, no splitting needed
    logs=True,
    wait=True
)

INFO:sagemaker:Creating transform job with name: wildscan-image-classifiers-2025-06-23-1-2025-06-23-20-11-06-335


....................................Docker entrypoint called with argument(s): serve
Running default environment configuration script
Nvidia gpu devices, drivers and cuda toolkit versions (only available on hosts with GPU):
Mon Jun 23 20:17:09 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.256.02   Driver Version: 470.256.02   CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            On   | 00000000:00:1E.0 Off |                    0 |
| N/A   34C    P8     8W /  70W |      0MiB / 15109MiB |      0%      Default |
|                       

In [12]:
# Run Evaluation Script on test set
image_uri = sagemaker.image_uris.retrieve(
    framework='sklearn',        # or 'xgboost', 'pytorch', etc.
    region=region,
    version='1.2-1',            # Specify the version you need
    py_version='py3',           # Specify Python version if required
       # Use 'processing' for processing jobs
)
print(image_uri)
# Define your processing container (can use a built-in or custom image)
script_processor1 = ScriptProcessor(
    command=['python3'],
    image_uri=image_uri,  # e.g., a scikit-learn or custom image
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    
)

# Run the processing job on the validation prediction set
s3_evaluation_out = f"{s3_test_path}/evaluation"
s3_true_meta_uri = f"{s3_test_path}/{test_key}"

script_processor1.run(
    code='../src/evaluation/evaluate.py',  # Your processing script
    inputs=[
        # S3 location of batch transform predictions files
        ProcessingInput(
            source=s3_transform_out,       # S3 bucket with predictions
            destination='/opt/ml/processing/input_predictions'        # Where the script will read input in local container
        ),
        
        # S3 location of the ground truth labels for the images in this set
        ProcessingInput(
            source=s3_true_meta_uri,
            destination='/opt/ml/processing/true_labels'
        ),

        # Label Mapping
        ProcessingInput(
            source=s3_label_map_uri,
            destination='/opt/ml/processing/label_mapping'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',           # Where the script will write output files in local container
            destination=s3_evaluation_out    # S3 bucket to store results
        )
    ]
)

INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3


INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2025-06-23-20-22-42-415


.............[2025-06-23 20:24:51.080955] Script has started.
INFO:root:Script has started at 2025-06-23 20:24:51.080986.
Number of files in /opt/ml/processing/input_predictions: 7932
pred probs df shape:(7932, 2)
 (step 1) pred probs df shape: (7932, 2)
 (step 2) True Labels data shape: (7932, 8)
 (step3) Merged data shape : (7932, 9)
 (step4) Merged data shape with preds: (7932, 11)
 (step5) METRICS CALCULATION
 (step5.1) Class-Restricted metrics - Evaluate only on labels available during training
 (step5.1) class_restriced df shape : (7804, 11)
/miniconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/miniconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill